# Voice under Stress - Analysis

import of all functions needed

In [ ]:
import os
import sys
import pandas as pd
import datetime as dt
import imageio
import numpy as np
import subprocess
import matplotlib.pyplot as plt
import seaborn as sns
import scipy 
import math
from statsmodels.stats.descriptivestats import sign_test

from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc, precision_recall_curve, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, LeaveOneOut, StratifiedKFold, StratifiedShuffleSplit, train_test_split
from sklearn import metrics, svm, linear_model
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler, MaxAbsScaler


from scipy import stats
from scipy.stats import skew, kurtosis
from statsmodels.stats.descriptivestats import sign_test

from matplotlib import rcParams

np.random.seed(42)

In [ ]:
homepath=os.getcwd()
scripts_path=os.path.join(homepath, 'scripts')
sys.path.append(scripts_path)
import myml
import mystress

In [ ]:
# set current dir to highest hierachy to add data path
os.chdir('/')
data_folder='/data'
sys.path.append(data_folder)
os.chdir(homepath)

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
filename_librosa='./processed_nemo_data/features.csv'
filename_behavior=data_folder + 'raw/TSST_behavior.csv'
filename_praat='./processed_nemo_data/new_praat_results'
filename_opensmile='./processed_nemo_data/opensmile_features.csv'
filename_output='./processed_nemo_data/df_vpn'

# Main Code

### Step 1: load data and calculate variables

In [ ]:
df, librosa_featurenames, praat_featurenames, opensimle_feature=mystress.load_df(filename_opensmile, filename_praat, 
                                                 filename_librosa, filename_behavior, filename_output)

In [ ]:
df=mystress.calc_var(df)
df_vpn = df.groupby("vpn").mean(numeric_only=True).reset_index()

### Step 2: Check the Dataset

Take a glimpse of the data using the desribe(), head() and info() methods 

In [ ]:
audio=praat_featurenames + list(opensimle_feature)+librosa_featurenames
df[audio].head()

In [ ]:
df[audio].describe(include = 'all')

### Checking the Stress Induction

In [ ]:
rcParams['figure.figsize'] = 8,6
plt.bar(df_vpn['Cond'].unique(), df_vpn['Cond'].value_counts(), color = ['red', 'green'])
plt.xticks([0, 1])
plt.xlabel('Target Classes')
plt.ylabel('Count')
plt.title('Count of each Target Class')
plt.show()

Let's check, whether the stressed people were stressed and the non-stressed were not:

Indicators of Stress: 

df['Cortisol_React']=(df.Cortisol_Max)-(df.cort_baseline) => positive

PANAS (Neg. Affect and Pos. Affect)

In [ ]:
print (df_vpn[((df_vpn['PANA_Delta_NA']<0) & (df_vpn.Cond==1))].vpn)
print (df_vpn.loc[(df_vpn['Cortisol_React']<0) & (df_vpn.Cond==1)].Cortisol_React)

In [ ]:
print (df_vpn.loc[(df_vpn['Cortisol_React']>5) & (df_vpn.Cond==0)].vpn)
print (df_vpn.loc[(df_vpn['PANA_Delta_NA']>0) & (df_vpn.Cond==0)].PANA_Delta_NA)

In [ ]:
# simulating a pandas df['type'] column
types = df_vpn.vpn
y_coords = df_vpn.PANA_Delta_NA 
x_coords = df_vpn.Cond #

for i,type in enumerate(types):
    x = x_coords[i]
    y = y_coords[i]
    plt.scatter(x, y, marker='x', color='red')
    plt.text(x+0.01, y+0.01, type, fontsize=20)
plt.show() 

In [ ]:
y_coords = df_vpn.Cortisol_React 

for i,type in enumerate(types):
    x = x_coords[i]
    y = y_coords[i]
    plt.scatter(x, y, marker='x', color='red')
    plt.text(x+0.01, y+0.01, type, fontsize=16)
plt.show() 

In [ ]:
df_vpn=df.groupby(['vpn']).mean(numeric_only=True).reset_index()
col_of_interest=['Cortisol_MinMax', 'Cortisol_React', 'sAA_React', 'PANA_Delta_NA', 'PANA_Delta_PA']
print (df_vpn.groupby(['Cond'])[col_of_interest].count())
print (df_vpn.groupby(['Cond'])[col_of_interest].mean())
print (df_vpn.groupby(['Cond'])[col_of_interest].std())

In [ ]:
col_of_interest=['LNCortisol_BL', 'LNCortisol_post1', 'LNCortisol_post20', 'cort_20_min']
print (df_vpn.groupby(['Cond'])[col_of_interest].count())
print (df_vpn.groupby(['Cond'])[col_of_interest].mean())
print (df_vpn.groupby(['Cond'])[col_of_interest].std())

In [ ]:
col_of_interest=['LNsAA_BL', 'LNsAA_post1', 'LNsAA_post20']
df_vpn.groupby(['Cond'])[col_of_interest].describe()